# Tăng tốc mô hình AssocRules trên CPU

Dùng sau notebook 06. Giữ nguyên K-means, số cụm, dữ liệu và ngưỡng; cache các lời gọi predict trùng và gộp xử lý cửa sổ. Chạy riêng mô hình cuối, giữ nguyên bốn mô hình đã chạy. Mỗi bước dùng output mới để giữ provenance. Bản benchmark nhỏ k=2 không đại diện thời gian full k=300.

In [1]:
from pathlib import Path
import subprocess, sys
PROJECT_HINT = Path('/content/drive/MyDrive/00_Research')
candidates = [PROJECT_HINT, Path.cwd(), *Path.cwd().parents]
CODE = next((p/'code' if (p/'code/comparison/config.json').exists() else p
             for p in candidates if (p/'code/comparison/config.json').exists() or
             (p/'comparison/config.json').exists()), None)
assert CODE is not None, 'Sửa PROJECT_HINT tới dự án đã chạy notebook 06'
CODE = CODE.resolve()
CACHE = Path('/content/ctt_compare_cache') if 'google.colab' in sys.modules else CODE/'results/comparison_data_v1'
FAST_OUTPUT = CODE/'results/assoc_exact_fast_v1'
# Sửa thành thư mục chứa kết quả bốn mô hình GPU của bạn.
BASE_OUTPUT = CODE/'results/comparison_gpu4_reuse_v1'
print('Cache:', CACHE, '\nFast CPU output:', FAST_OUTPUT, '\nGPU baseline output:', BASE_OUTPUT)
def run_fast(stage, output):
    subprocess.run([sys.executable, str(CODE/'scripts/run_comparison_fast.py'), '--stage', stage,
                    '--methods', 'assoc_rules_crysys', '--cache', str(CACHE), '--output', str(output)], check=True)


Cache: F:\00_Research\code\results\comparison_data_v1 
Fast CPU output: F:\00_Research\code\results\assoc_exact_fast_v1 
GPU baseline output: F:\00_Research\code\results\comparison_gpu4_reuse_v1


In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(CODE/'tests/test_assoc_acceleration.py'), str(CODE/'tests/test_merge_comparison_parts.py')], cwd=CODE.parent, check=True)

CompletedProcess(args=['F:\\00_Research\\.venv\\Scripts\\python.exe', '-m', 'pytest', '-q', 'F:\\00_Research\\code\\tests\\test_assoc_acceleration.py', 'F:\\00_Research\\code\\tests\\test_merge_comparison_parts.py'], returncode=0)

In [3]:
subprocess.run([sys.executable, str(CODE/'scripts/benchmark_assoc_speed.py'), '--cache', str(CACHE), '--output', str(CODE/'results/assoc_speed_check_local.json')], check=True)

CompletedProcess(args=['F:\\00_Research\\.venv\\Scripts\\python.exe', 'F:\\00_Research\\code\\scripts\\benchmark_assoc_speed.py', '--cache', 'F:\\00_Research\\code\\results\\comparison_data_v1', '--output', 'F:\\00_Research\\code\\results\\assoc_speed_check_local.json'], returncode=0)

## Pilot và chạy riêng AssocRules
Pilot chỉ dùng nguồn, một cấu hình nhỏ để kiểm tra. Full run vẫn giữ k=300 và ba seed như cấu hình gốc. Nếu bị ngắt, gọi lại cùng lệnh; cell COMPLETE được kiểm tra rồi bỏ qua.

In [4]:
run_fast('pilot', CODE/'results/assoc_exact_fast_pilot')

In [5]:
run_fast('run', FAST_OUTPUT)

## Gộp với bốn mô hình đã chạy
Chỉ chạy khi BASE_OUTPUT và FAST_OUTPUT có đủ set/seed. Công cụ đọc metric và kiểm tra hash; giữ nguyên manifest gốc. Config, input và môi trường phải khớp, chỉ cho phép khác backend cache đã khai báo. Không dùng kết quả pilot hoặc gộp cả hai bản native/fast của cùng một cell.

In [6]:
# Đợi notebook GPU hoàn tất trước khi gộp; lỗi GPU được báo rõ.
import time,json
GPU_STATUS=CODE/'notebooks/07_gpu_four_models_executed.execution.json'
while True:
    state=json.loads(GPU_STATUS.read_text()) if GPU_STATUS.exists() else {}
    if state.get('status')=='COMPLETE':break
    if state.get('status')=='FAILED':raise RuntimeError('Notebook GPU gặp lỗi: '+state.get('error',''))
    print('Đang đợi đủ 48 cell GPU; AssocRules đã hoàn tất.',flush=True)
    time.sleep(30)
MERGED = CODE/'results/comparison_merged_fast'
subprocess.run([sys.executable, str(CODE/'scripts/merge_comparison_parts.py'), '--inputs', str(BASE_OUTPUT), str(FAST_OUTPUT), '--output', str(MERGED)], check=True)
import pandas as pd
print((MERGED/'coverage.json').read_text())
display(pd.read_csv(MERGED/'paired_by_set.csv'))

Đang đợi đủ 48 cell GPU; AssocRules đã hoàn tất.


Đang đợi đủ 48 cell GPU; AssocRules đã hoàn tất.


Đang đợi đủ 48 cell GPU; AssocRules đã hoàn tất.


Đang đợi đủ 48 cell GPU; AssocRules đã hoàn tất.


Đang đợi đủ 48 cell GPU; AssocRules đã hoàn tất.


Đang đợi đủ 48 cell GPU; AssocRules đã hoàn tất.


... (252 outputs left out)

Đang đợi đủ 48 cell GPU; AssocRules đã hoàn tất.


Đang đợi đủ 48 cell GPU; AssocRules đã hoàn tất.


{
  "status": "COMPLETE_MATRIX",
  "cells": 60,
  "methods": [
    "assoc_rules_crysys",
    "canbertplus",
    "canlp",
    "raw_transformer",
    "structural_transformer"
  ],
  "analysis_code_sha256": "a7ee0e2f1c5ebd6adb8357a4a16f39797eff9b79d698d561aef8d3d4ec75becc",
  "note": "Original execution manifests retained. Native code/data/config/environment must match; only declared exact-call caching backend may differ."
}


,opponent,condition,set,Recall,FAR_per_100k,Attack_F1,AP
0,assoc_rules_crysys,A,set_01,0.455864,42.579217,0.614627,0.602722
1,assoc_rules_crysys,A,set_02,0.363825,115.090932,0.488187,0.026758
2,assoc_rules_crysys,A,set_03,0.719079,88.286359,0.828813,0.330088
3,assoc_rules_crysys,A,set_04,0.815606,80.706342,0.897063,0.698624
4,assoc_rules_crysys,B,set_01,0.494978,32.136345,0.587030,0.767838
...,...,...,...,...,...,...,...
75,raw_transformer,D,set_04,-0.217260,-57058.413349,0.221937,0.315775
76,raw_transformer,validation,set_01,-0.080087,-30.147858,-0.037792,-0.092737
77,raw_transformer,validation,set_02,-0.058707,0.275341,-0.068027,-0.031630
78,raw_transformer,validation,set_03,-0.041405,27.287257,-0.056362,0.008187
